https://deepeval.com/guides/guides-using-custom-llms
https://deepeval.com/integrations/models/ollama


In [2]:
 !ollama list

NAME               ID              SIZE      MODIFIED      
qwen3:1.7b         8f68893c685c    1.4 GB    3 days ago       
llama3.2:1b        baf6a787fdff    1.3 GB    6 days ago       
qwen3:0.6b         7df6b6e09427    522 MB    3 months ago     
qwen3:latest       500a1f067a9f    5.2 GB    3 months ago     
qwen2.5:latest     845dbda0ea48    4.7 GB    8 months ago     
llama3.2:latest    a80c4f17acd5    2.0 GB    11 months ago    


In [1]:
!deepeval set-ollama qwen3:1.7b

Settings updated for this session. To persist, use --save=dotenv[:path] (default
.env.local) or set DEEPEVAL_DEFAULT_SAVE=dotenv:.env.local
🙌 Congratulations! You're now using a local Ollama model `qwen3:1.7b` for all 
evals that require an LLM.


In [13]:
import os
from ollama import Client

key = '97b003f43e5e4d89a0444272828242c7.BMYErA3vAD-dNMfVQSOhyXai'
client = Client(
    host="https://ollama.com",
    headers={'Authorization': 'Bearer ' + key})

messages = [
  {
    'role': 'user',
    'content': 'Why is the sky blue?',
  },
]
response = ''
for part in client.chat('gpt-oss:120b', messages=messages, stream=True):
    # if part.choices[0].delta.content is not None:
    #     response += part.choices[0].delta.content
    #print('part')
    # answer += part['message']['content']
    #print(part['message']['content'], end='', flush=True)
    response += part['message']['content']
newl = response.find('\n')
print(response[:newl])

The sky looks blue because sunlight is scattered by the gases and tiny particles in Earth’s atmosphere, and blue light is scattered much more than the other colors.


In [25]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams, LLMTestCase
from deepeval.models import OllamaModel
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval import evaluate

model_name = "qwen3:1.7b"
#model_name = "deepseek-v3.1:671b-cloud"
#model_name="qwen3-coder:480b-cloud"
model = OllamaModel(
    # host="https://ollama.com",
    # headers={'Authorization': 'Bearer ' + key}
    model=model_name,
    base_url="http://localhost:11434",
    temperature=0.1
)

answer_relevancy = AnswerRelevancyMetric(model=model)

In [26]:
#!deepeval set-ollama qwen3-coder:480b-cloud
!deepeval set-ollama qwen3:1.7b

test_case = LLMTestCase(input="What do you like", actual_output="I love cats")
evaluate(test_cases=[test_case], metrics=[answer_relevancy])

zsh:1: unmatched "


✨ You're running DeepEval's latest Answer Relevancy Metric! (using qwen3:1.7b (Ollama), strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: qwen3:1.7b (Ollama), reason: The score is 1.00 because the output directly addresses the question without any irrelevant statements. The JSON structure is concise and relevant to the input query., error: None)

For test case:

  - input: What do you like
  - actual output: I love cats
  - expected output: None
  - context: None
  - retrieval context: None


Overall Metric Pass Rates

Answer Relevancy: 100.00% pass rate




⚠ WARNING: No hyperparameters logged.
» ]8;id=713394;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 160.85s | token cost: 0.0 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=1.0, reason='The score is 1.00 because the output directly addresses the question without any irrelevant statements. The JSON structure is concise and relevant to the input query.', strict_mode=False, evaluation_model='qwen3:1.7b (Ollama)', error=None, evaluation_cost=0.0, verbose_logs='Statements:\n[\n    "I",\n    "love",\n    "cats"\n] \n \nVerdicts:\n[\n    {\n        "verdict": "idk",\n        "reason": "ambiguous"\n    },\n    {\n        "verdict": "idk",\n        "reason": "ambiguous"\n    },\n    {\n        "verdict": "yes",\n        "reason": "relevant"\n    }\n]')], conversational=False, multimodal=False, input='What do you like', actual_output='I love cats', expected_output=None, context=None, retrieval_context=None, turns=None, additional_metadata=None)], confident_link=None, test_run_id=None)

In [30]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams, LLMTestCase

criteria = """Coherence (1-5) - the collective quality of all sentences. We align this dimension with
the DUC quality question of structure and coherence whereby the summary should be
well-structured and well-organized. The summary should not just be a heap of related information, but should build from sentence to sentence to a coherent body of information about a topic."""

coherence_metric = GEval(
    name="Coherence",
    model=model,
    criteria=criteria,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
)

# Now define your test case, actual_output is your LLM output
test_case = LLMTestCase(input="Hey how's the weather like today?", actual_output="It's alright!", expected_output="Rainy")

# Use G-Eval metric
coherence_metric.measure(test_case)
print(coherence_metric.score, coherence_metric.reason)

Output()

0.0 The actual output 'It's alright!' does not match the expected 'Rainy' in content and structure. The sentence flow and logical progression are inconsistent, failing to align with the expected theme of weather description. The coherence and thematic unity are completely disrupted.


In [24]:
#https://medium.com/@jeffreyip54/you-can-now-use-ollama-for-llm-as-a-judge-76f06e3005c9

#from deepeval.dataset import EvaluationDataset

from deepeval.metrics import (
  ContextualRelevancyMetric,
  ContextualRecallMetric,
  ContextualPrecisionMetric,
#  AnswerRelevnacyMetric,
  FaithfulnessMetric
)

contextual_precision = ContextualPrecisionMetric(model=model)
contextual_recall = ContextualRecallMetric(model=model)
contextual_relevancy = ContextualRelevancyMetric(model=model)
answer_relevancy = AnswerRelevancyMetric(threshold=0.8, model=model)
faithfulness = FaithfulnessMetric(model=model)

#dataset = EvaluationDataset()

evaluate(test_cases=[test_case], metrics=[contextual_precision, contextual_recall, contextual_relevancy, answer_relevancy, faithfulness])

✨ You're running DeepEval's latest Contextual Precision Metric! (using qwen3:1.7b (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using qwen3:1.7b (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using qwen3:1.7b (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using qwen3:1.7b (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using qwen3:1.7b (Ollama), strict=False, 
async_mode=True)...

Output()

MissingTestCaseParamsError: 'retrieval_context' and 'expected_output' cannot be None for the 'Contextual Precision' metric

In [19]:
host="https://ollama.com",
headers={'Authorization': 'Bearer ' + key}
model = 'gpt-oss:120b'
!deepeval set-local-model=model  --base-url=host --api-key=key

Usage: deepeval [OPTIONS] COMMAND [ARGS]...
Try 'deepeval --help' for help.
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ No such command 'set-local-model=model'.                                     │
╰──────────────────────────────────────────────────────────────────────────────╯
